# STAGE 1

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted!")

Mounted at /content/drive
Drive mounted!


In [ ]:
# ── Cell 2: Verify GPU ──────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# ── Cell 3: Define Paths ────────────────────────────────────────────────────
import os

# 🔹 USER CONFIGURATION ─────────────────────────────────────────────
# Set your project root directory
BASE_DIR = "/path_to_project_root"   # <- change this

# Choose model: "mobilebert", "albert", "tinybert"
MODEL_NAME = "mobilebert"
# ──────────────────────────────────────────────────────────────────


# ── Derived paths ──
CODE_DIR    = os.path.join(BASE_DIR, f"gector_code_{MODEL_NAME}")
DATA_DIR    = os.path.join(BASE_DIR, "gec_gector_datasets_new")
MODELS_ROOT = os.path.join(BASE_DIR, "models")

# ── Runtime working dir ──
WORK_DIR = os.path.join(BASE_DIR, "work")

# ── Dataset files ──
PIE_STAGE1 = os.path.join(DATA_DIR, "pie_a5_gector.txt")
LANG8      = os.path.join(DATA_DIR, "lang8_gector.txt")
NUCLE      = os.path.join(DATA_DIR, "nucle_gector.txt")
FCE        = os.path.join(DATA_DIR, "fce_train_gector.txt")
WI_TRAIN   = os.path.join(DATA_DIR, "wi_train_gector.txt")
WI_DEV     = os.path.join(DATA_DIR, "wi_dev_gector.txt")

# ── Model output dirs ──
STAGE1_DIR = os.path.join(MODELS_ROOT, "stage1", MODEL_NAME)
STAGE2_DIR = os.path.join(MODELS_ROOT, "stage2", MODEL_NAME)
STAGE3_DIR = os.path.join(MODELS_ROOT, "stage3", MODEL_NAME)

# ── Vocab path ──
VOCAB_PATH = os.path.join(CODE_DIR, "data", "output_vocabulary")

# ── Create directories ──
for d in [STAGE1_DIR, STAGE2_DIR, STAGE3_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Info print ──
print("✅ Paths defined!")
print(f"  Base Dir:   {BASE_DIR}")
print(f"  Model:      {MODEL_NAME}")
print(f"  Code Dir:   {CODE_DIR}")
print(f"  Data Dir:   {DATA_DIR}")
print(f"  Models Dir: {MODELS_ROOT}")


✅ Paths defined!
  Code:   /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code_mobilebert
  Data:   /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new
  Models: /content/drive/Shareddrives/gec_thesis/gec_thesis/models


In [ ]:
# ── Cell 4: Verify Drive Files ──────────────────────────────────────────────
import os

print("Verifying dataset files:")
for name, path in [
    ("PIE Stage1", PIE_STAGE1),
    ("WI Dev",     WI_DEV),
    ("Vocab",      VOCAB_PATH),
]:
    exists = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  {exists}  {name}: {path}")


Verifying dataset files:
  ✅  PIE Stage1: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/pie_a5_gector.txt
  ✅  WI Dev: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  ✅  Vocab: /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code_mobilebert/data/output_vocabulary


In [ ]:
# ── Cell 5: Copy Code to /content/gector ────────────────────────────────────
import shutil

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DIR, WORK_DIR)
print("✅ Code copied to:", WORK_DIR)


✅ Code copied to: /content/gector


In [ ]:
# ── Cell 6: Install Dependencies ────────────────────────────────────────────
import os
os.system("pip install transformers torch tqdm python-Levenshtein -q")
print("✅ Dependencies installed!")


✅ Dependencies installed!


In [ ]:
# ── Cell 7: Cache Encoder Models ─────────────────────────────────────────────
from transformers import AutoModel, AutoTokenizer
import torch

print("Pre-downloading all encoder models...")

for model_name in [
    "huawei-noah/TinyBERT_General_4L_312D",
    "albert-base-v2",
    "google/mobilebert-uncased"
]:
    print(f"Downloading {model_name}...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  Done!")

torch.cuda.empty_cache()
print("\n✅ All models cached! Training will not need internet.")


Pre-downloading all encoder models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertModel LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
mobilebert.embeddings.position_ids         | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.dense.weight               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Done!

✅ All models cached! Training will not need internet.


In [ ]:
# ── Cell 8: Run Stage 1 Training ─────────────────────────────────────────────
import subprocess
import os
import sys

# ⚠️ IMPORTANT:
# The transformer model is controlled by MODEL_NAME.
# To switch models, simply change:
#   MODEL_NAME = "mobilebert"  → "albert" or "tinybert"
# No need to modify anything in this cell.

os.chdir(WORK_DIR)

stage1_cmd = [
    "python", "resume_training.py",
    "--train_set",          PIE_STAGE1,
    "--dev_set",            WI_DEV,
    "--model_dir",          STAGE1_DIR,
    "--vocab_path",         VOCAB_PATH,
    "--transformer_model",  MODEL_NAME,  # uses selected model
    "--batch_size",         "128",
    "--n_epoch",            "20",
    "--patience",           "10",
    "--cold_steps_count",   "2",
    "--cold_lr",            "1e-3",
    "--lr",                 "5e-5",
    "--skip_correct",       "1",
    "--skip_complex",       "0",
    "--tune_bert",          "1",
    "--tag_strategy",       "keep_one",
    "--lowercase_tokens",   "1",
    "--pieces_per_token",   "5",
    "--tn_prob",            "0.0",
    "--tp_prob",            "1.0",
    "--updates_per_epoch",  "10000",
    "--special_tokens_fix", "0",
    "--accumulation_size",  "8",
]

IMPORTANT_KEYWORDS = [
    "Using encoder", "Vocab loaded", "Train:", "Dev:",
    "Pretrained weights loaded", "Using device",
    "RESUMING", "Resuming", "Training on device",
    "Starting from epoch", "Total epochs", "Batch size",
    "Effective batch", "Updates/epoch", "Patience",
    "EPOCH", "Train loss", "Val loss", "Best epoch",
    "Saved", "improvement", "Patience:", "Summary",
    "TRAINING COMPLETE", "Best val loss", "Total time",
]

print(f"Starting Stage 1 Training ({MODEL_NAME})...")
print(f"  Train set : {PIE_STAGE1}")
print(f"  Dev set   : {WI_DEV}")
print(f"  Model dir : {STAGE1_DIR}")
print(f"  Vocab     : {VOCAB_PATH}")
print()

process = subprocess.Popen(
    stage1_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    line = line.rstrip()
    if "Training epoch" in line:
        sys.stdout.write(f"\r{line}")
        sys.stdout.flush()
    elif any(kw in line for kw in IMPORTANT_KEYWORDS):
        sys.stdout.write(f"\n{line}\n")
        sys.stdout.flush()

process.wait()
print("\n✅ Stage 1 Done!")

Starting Stage 1 Training (MobileBERT)...
  Train set : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/pie_a5_gector.txt
  Dev set   : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  Model dir : /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert
  Vocab     : /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code_mobilebert/data/output_vocabulary


Using encoder: google/mobilebert-uncased

Vocab loaded: 5002 labels, 4 d_tags

Train: 8865347 instances | Dev: 4382 instances

Using device: cuda:0

Training on device  : cuda:0

Starting from epoch : 1

Total epochs        : 20

Batch size          : 128

Effective batch     : 1024

Updates/epoch       : 10000

Patience            : 10

EPOCH 1/20
Training epoch 1:  14%|█▍        | 10000/68992 [39:35<3:53:36,  4.21it/s, loss=2.4901, updates=1250]
Train loss : 2.4901

Val loss   : 1.4312

💾 Saved 🏆 BEST (epoch 1) → model_best.th

💾 Saved e

In [14]:
# ── Cell 9: Zip Stage 1 Best Model for Stage 2 ──────────────────────────────
import zipfile
import os

zip_path = f"{STAGE1_DIR}/stage1_mobilebert_best.zip"

files_to_zip = [
    ("model_best.th",        f"{STAGE1_DIR}/model_best.th"),
    ("training_history.json",f"{STAGE1_DIR}/training_history.json"),
    ("metrics.json",         f"{STAGE1_DIR}/metrics.json"),
]

print("Zipping Stage 1 best model...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arcname, filepath in files_to_zip:
        if os.path.exists(filepath):
            zf.write(filepath, arcname)
            size = os.path.getsize(filepath) / 1e6
            print(f"  ✅ {arcname} ({size:.1f} MB)")
        else:
            print(f"  ❌ {arcname} — NOT FOUND, skipping")

print(f"\n✅ Zip saved to: {zip_path}")

Zipping Stage 1 best model...
  ✅ model_best.th (109.1 MB)
  ✅ training_history.json (0.0 MB)
  ✅ metrics.json (0.0 MB)

✅ Zip saved to: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/stage1_mobilebert_best.zip


# STAGE 2

In [ ]:
# ── Cell 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 2: Verify GPU ──────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# ── Cell 3: Paths ──
import os

# 🔹 USER CONFIG ─────────────────────────────────────────────
# Set your project root directory
BASE_DIR = "/path_to_project_root"   # <- change this

# Choose model: "mobilebert", "albert", "tinybert"
MODEL_NAME = "mobilebert"
# ───────────────────────────────────────────────────────────


# ── Derived directories ──
DATA_DIR    = os.path.join(BASE_DIR, "gec_gector_datasets_new")
CODE_DIR    = os.path.join(BASE_DIR, f"gector_code_{MODEL_NAME}")
MODELS_ROOT = os.path.join(BASE_DIR, "models")

# ── Stage checkpoints ──
STAGE1_CKPT = os.path.join(MODELS_ROOT, "stage1", MODEL_NAME)
STAGE2_OUT  = os.path.join(MODELS_ROOT, "stage2", MODEL_NAME)

# ── Runtime working directory ──
WORK_DIR = os.path.join(BASE_DIR, "work")

# ── Local staging dirs (runtime safe copies) ──
STAGE1_DIR = os.path.join(WORK_DIR, "stage1", MODEL_NAME)
STAGE2_DIR = os.path.join(WORK_DIR, "stage2", MODEL_NAME)

# ── Create required directories ──
for d in [WORK_DIR, STAGE1_DIR, STAGE2_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Dataset paths ──
WI_DEV = os.path.join(DATA_DIR, "wi_dev_gector.txt")
stage2_train = os.path.join(DATA_DIR, "stage2_train.txt")

# ── Vocab path ──
VOCAB_PATH = os.path.join(CODE_DIR, "data", "output_vocabulary")

# ── Debug info ──
print("Stage 2 Paths Defined")
print(f"  Base Dir     : {BASE_DIR}")
print(f"  Model        : {MODEL_NAME}")
print(f"  Code Dir     : {CODE_DIR}")
print(f"  Data Dir     : {DATA_DIR}")
print(f"  Stage1 CKPT  : {STAGE1_CKPT}")
print(f"  Stage2 OUT   : {STAGE2_OUT}")
print(f"  Train file   : {stage2_train}")
print(f"  Dev file     : {WI_DEV}")

✅ Paths defined (FIXED)
   DRIVE_ROOT  : /content/drive/Shareddrives/gec_thesis/gec_thesis
   CODE_DIR    : /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code_mobilebert
   STAGE1_CKPT : /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert
   STAGE2_OUT  : /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage2/mobilebert
   stage2_train: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/stage2_train.txt


In [ ]:
# ── Cell 4: Verify all input files (FINAL CLEAN VERSION) ──
import os

def check_path(name, path):
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {name}")
    print(f"   Path: {path}")

    if not exists:
        parent = os.path.dirname(path)
        print(f"   🔍 Checking parent: {parent}")

        if os.path.exists(parent):
            try:
                files = os.listdir(parent)
                print(f"   📂 Found {len(files)} items. Sample:")
                for f in files[:10]:
                    print(f"      - {f}")
            except Exception as e:
                print(f"   ⚠️ Cannot list directory: {e}")
        else:
            print("   ❌ Parent directory does NOT exist")
    print()

checks = [
    ("W&I Dev", WI_DEV),
    ("stage2_train", stage2_train),
    ("Stage1 model_best.th", os.path.join(STAGE1_CKPT, "model_best.th")),
    ("Stage1 metrics.json", os.path.join(STAGE1_CKPT, "metrics.json")),
    ("Stage1 training_history.json", os.path.join(STAGE1_CKPT, "training_history.json")),
    ("Code dir", CODE_DIR),
]

all_ok = True
for name, path in checks:
    if not os.path.exists(path):
        all_ok = False
    check_path(name, path)

print("✅ All files found — ready to proceed!" if all_ok else "❌ Fix missing files above before continuing.")

✅ W&I Dev
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt

✅ stage2_train
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/stage2_train.txt

✅ Stage1 model_best.th
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/model_best.th

✅ Stage1 metrics.json
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/metrics.json

✅ Stage1 training_history.json
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/training_history.json

✅ Code dir
   Path: /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code_mobilebert

✅ All files found — ready to proceed!


In [ ]:
# ── Cell 5: Copy gector_code_mobilebert to local working dir ──
import shutil, os

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DIR, WORK_DIR)
print("✅ Code copied to", WORK_DIR)

for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        print(os.path.join(root, f).replace(WORK_DIR, ""))

✅ Code copied to /content/gector
/README_MOBILEBERT.md
/requirements.txt
/predict.py
/train.py
/resume_training.py
/utils/prepare_clc_fce_data.py
/utils/preprocess_data.py
/utils/helpers.py
/data/verb-form-vocab.txt
/data/output_vocabulary/d_tags.txt
/data/output_vocabulary/labels.txt
/data/output_vocabulary/non_padded_namespaces.txt
/gector/gec_model.py
/gector/seq2labels_model.py
/gector/trainer.py
/gector/datareader.py


In [ ]:
# ── Cell 6: Restore Stage 1 checkpoint to local STAGE1_DIR ──
import shutil, os, json

files_to_restore = ["model_best.th", "training_history.json", "metrics.json"]

print("Restoring Stage 1 MOBILEBERT checkpoint...")
for fname in files_to_restore:
    src = os.path.join(STAGE1_CKPT, fname)
    dst = os.path.join(STAGE1_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f"  ✅ {fname} ({size:.1f} MB)")
    else:
        print(f"  ⚠️  {fname} not found — skipping")

# Show Stage 1 metrics
metrics_path = os.path.join(STAGE1_DIR, "metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("\nStage 1 final metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

Restoring Stage 1 MOBILEBERT checkpoint...
  ✅ model_best.th (109.1 MB)
  ✅ training_history.json (0.0 MB)
  ✅ metrics.json (0.0 MB)

Stage 1 final metrics:
  training_duration: 19:06:38
  best_epoch: 3
  best_val_loss: 1.1942
  last_epoch: 13
  total_epochs_trained: 13
  history: [{'epoch': 1, 'train_loss': 2.490121, 'val_loss': 1.431185, 'is_best': True, 'epoch_time_seconds': 2623, 'early_stopped': False}, {'epoch': 2, 'train_loss': 1.910097, 'val_loss': 1.393469, 'is_best': True, 'epoch_time_seconds': 2379, 'early_stopped': False}, {'epoch': 3, 'train_loss': 0.528843, 'val_loss': 1.194154, 'is_best': True, 'epoch_time_seconds': 5820, 'early_stopped': False}, {'epoch': 4, 'train_loss': 0.447339, 'val_loss': 1.198601, 'is_best': False, 'epoch_time_seconds': 5840, 'early_stopped': False}, {'epoch': 5, 'train_loss': 0.418209, 'val_loss': 1.201157, 'is_best': False, 'epoch_time_seconds': 5508, 'early_stopped': False}, {'epoch': 6, 'train_loss': 0.399069, 'val_loss': 1.207938, 'is_best': 

In [ ]:
# ── Cell 7: Check stage2_train.txt ──
import os

if os.path.exists(stage2_train):
    with open(stage2_train) as f:
        count = sum(1 for _ in f)
    print(f"✅ stage2_train.txt found: {count:,} lines")
else:
    print(f"❌ stage2_train.txt NOT found at: {stage2_train}")
    print("   Update the path in Cell 3 or upload the file to Drive.")

✅ stage2_train.txt found: 2,047,553 lines


In [ ]:
# ── Cell 8: Install Dependencies ────────────────────────────────────────────
import os
os.system("pip install transformers torch tqdm python-Levenshtein -q")
print("✅ Dependencies installed!")


✅ Dependencies installed!


In [ ]:
# ── Cell 9: Cache Encoder Models ─────────────────────────────────────────────
from transformers import AutoModel, AutoTokenizer
import torch

print("Pre-downloading all encoder models...")

for model_name in [
    "huawei-noah/TinyBERT_General_4L_312D",
    "albert-base-v2",
    "google/mobilebert-uncased"
]:
    print(f"Downloading {model_name}...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  Done!")

torch.cuda.empty_cache()
print("\n✅ All models cached! Training will not need internet.")


Pre-downloading all encoder models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertModel LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.dense.weight               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
mobilebert.embeddings.position_ids         | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Done!

✅ All models cached! Training will not need internet.


In [ ]:
# ── Cell 10: Run Stage 2 Training ─────────────────────────────────────────────
import subprocess
import os
import sys

# ⚠️ IMPORTANT:
# The transformer model is controlled by MODEL_NAME.
# To switch models, change only:
#   MODEL_NAME = "mobilebert" → "albert" or "tinybert"
# No need to modify this training cell.

os.chdir(WORK_DIR)
os.makedirs(STAGE2_DIR, exist_ok=True)

stage2_cmd = [
    "python", "resume_training.py",
    "--train_set",          stage2_train,
    "--dev_set",            WI_DEV,
    "--model_dir",          STAGE2_DIR,
    "--pretrain_folder",    STAGE1_DIR,
    "--pretrain",           "model_best",
    "--vocab_path",         VOCAB_PATH,
    "--transformer_model",  MODEL_NAME,  # uses selected model
    "--batch_size",         "32",
    "--accumulation_size",  "4",
    "--n_epoch",            "20",
    "--patience",           "5",
    "--cold_steps_count",   "0",
    "--lr",                 "2e-5",
    "--skip_correct",       "0",
    "--skip_complex",       "1",
    "--tune_bert",          "1",
    "--tag_strategy",       "keep_one",
    "--lowercase_tokens",   "1",
    "--pieces_per_token",   "5",
    "--tn_prob",            "0.5",
    "--tp_prob",            "0.7",
    "--updates_per_epoch",  "7000",
    "--special_tokens_fix", "0",
]

IMPORTANT_KEYWORDS = [
    "Using encoder", "Vocab loaded", "Train:", "Dev:",
    "Pretrained weights loaded", "Using device",
    "RESUMING", "Resuming", "Training on device",
    "Starting from epoch", "Total epochs", "Batch size",
    "Effective batch", "Updates/epoch", "Patience",
    "EPOCH", "Train loss", "Val loss", "Best epoch",
    "Saved", "improvement", "Patience:", "Summary",
    "TRAINING COMPLETE", "Best val loss", "Total time",
]

print(f"🚀 Starting Stage 2 Training ({MODEL_NAME})...")
print(f"  Train set   : {stage2_train}")
print(f"  Dev set     : {WI_DEV}")
print(f"  Stage1 ckpt : {STAGE1_DIR}")
print(f"  Model dir   : {STAGE2_DIR}")
print()

process = subprocess.Popen(
    stage2_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    line = line.rstrip()
    if "Training epoch" in line:
        sys.stdout.write(f"\r{line}")
        sys.stdout.flush()
    elif any(kw in line for kw in IMPORTANT_KEYWORDS):
        sys.stdout.write(f"\n{line}\n")
        sys.stdout.flush()

process.wait()
print("\n✅ Stage 2 Done!")

Starting Stage 2 Training (MOBILEBERT)...
  Train set  : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/stage2_train.txt
  Dev set    : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  Stage1 ckpt: /content/models/stage1/mobilebert
  Model dir  : /content/models/stage2/mobilebert


Using encoder: google/mobilebert-uncased

Vocab loaded: 5002 labels, 4 d_tags

Train: 2047553 instances | Dev: 4382 instances

Pretrained weights loaded!

Using device: cuda:0

Training on device  : cuda:0

Starting from epoch : 1

Total epochs        : 20

Batch size          : 32

Effective batch     : 128

Updates/epoch       : 7000

Patience            : 5

EPOCH 1/20
Training epoch 1:  19%|█▊        | 7000/37687 [14:19<1:02:48,  8.14it/s, loss=0.6804, updates=1750]
Train loss : 0.6804

Val loss   : 0.5902

💾 Saved 🏆 BEST (epoch 1) → model_best.th

💾 Saved epoch 1 → model_epoch1.th

💾 Saved last (epoch 1) → model_last.th

📊 Epoch 1 S

In [ ]:
# ── Cell 13: Zip Stage 2 Best Model for Stage 3 (SAVE TO DRIVE) ──────────────
import zipfile
import os

# Ensure output directory exists
os.makedirs(STAGE2_OUT, exist_ok=True)

zip_path = f"{STAGE2_OUT}/stage2_mobilebert_best.zip"

files_to_zip = [
    ("model_best.th",         f"{STAGE2_DIR}/model_best.th"),
    ("training_history.json", f"{STAGE2_DIR}/training_history.json"),
    ("metrics.json",          f"{STAGE2_DIR}/metrics.json"),
]

print("Zipping Stage 2 best model (saving to Drive)...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arcname, filepath in files_to_zip:
        if os.path.exists(filepath):
            zf.write(filepath, arcname)
            size = os.path.getsize(filepath) / 1e6
            print(f"  ✅ {arcname} ({size:.1f} MB)")
        else:
            print(f"  ❌ {arcname} — NOT FOUND, skipping")

print(f"\n✅ Zip saved to: {zip_path}")

Zipping Stage 2 best model (saving to Drive)...
  ✅ model_best.th (109.1 MB)
  ✅ training_history.json (0.0 MB)
  ✅ metrics.json (0.0 MB)

✅ Zip saved to: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage2/mobilebert/stage2_mobilebert_best.zip


# STAGE 3

In [ ]:
# ── Optional: Google Drive Mount (Colab only) ─────────────────────────────
# ⚠️ Use this ONLY if running in Google Colab

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = "/content/drive/Shareddrives/gec_thesis/gec_thesis"

    print("=== Checking Drive files ===")

    import os
    for root, dirs, files in os.walk(DRIVE_ROOT):
        for file in files:
            print(os.path.join(root, file))

except ImportError:
    print("Not running in Colab. Skipping Drive mount.")

Mounted at /content/drive
=== Checking Drive files ===
/content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_train_gector.txt
/content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
/content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/pie_a5_gector.txt
/content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/stage2_train.txt
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/model_best.th
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/metrics.json
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/training_history.json
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/stage1_mobilebert_best.zip
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/vocabulary/labels.txt
/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage1/mobilebert/vocabulary/d_tags.txt
/conte

In [ ]:
# ── Optional: Copy Code to Runtime Workspace (Colab only) ────────────────
import os
import shutil

# ⚠️ Runs only in Google Colab
try:
    from google.colab import drive

    # 🔹 Use your project config instead of hardcoded Drive path
    BASE_DIR = "/path_to_project_root"
    MODEL_NAME = "mobilebert"

    DRIVE_ROOT = BASE_DIR

    code_path = os.path.join(DRIVE_ROOT, f"gector_code_{MODEL_NAME}")
    work_dir = os.path.join("/content", "gector")

    # Recreate working directory
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)

    shutil.copytree(code_path, work_dir)

    print("✅ Code copied to runtime!")

    # Optional: show file structure
    for root, dirs, files in os.walk(work_dir):
        for f in files:
            print(os.path.join(root, f))

except ImportError:
    print("Not running in Colab. Skipping code copy step.")

✅ Code copied!
/content/gector/README_MOBILEBERT.md
/content/gector/requirements.txt
/content/gector/predict.py
/content/gector/train.py
/content/gector/resume_training.py
/content/gector/utils/prepare_clc_fce_data.py
/content/gector/utils/preprocess_data.py
/content/gector/utils/helpers.py
/content/gector/data/verb-form-vocab.txt
/content/gector/data/output_vocabulary/d_tags.txt
/content/gector/data/output_vocabulary/labels.txt
/content/gector/data/output_vocabulary/non_padded_namespaces.txt
/content/gector/gector/gec_model.py
/content/gector/gector/seq2labels_model.py
/content/gector/gector/trainer.py
/content/gector/gector/datareader.py


In [3]:
import os
os.system("pip install transformers torch tqdm python-Levenshtein sentencepiece -q")
print("✅ Dependencies installed!")

✅ Dependencies installed!


In [ ]:
import os

# 🔹 USER CONFIG ─────────────────────────────────────────────
# Change only these two for different setups/models
BASE_DIR = "/path_to_project_root"

# Models: "mobilebert", "albert", "tinybert"
MODEL_NAME = "mobilebert"
# ───────────────────────────────────────────────────────────


# ── Paths ──
DATA_DIR    = os.path.join(BASE_DIR, "gec_gector_datasets_new")
CODE_DIR    = os.path.join(BASE_DIR, f"gector_code_{MODEL_NAME}")
MODELS_ROOT = os.path.join(BASE_DIR, "models")

# ── Runtime workspace ──
WORK_DIR = os.path.join(BASE_DIR, "work")

# ── Dataset files ──
WI_TRAIN = os.path.join(DATA_DIR, "wi_train_gector.txt")
WI_DEV   = os.path.join(DATA_DIR, "wi_dev_gector.txt")

# ── Vocab path ──
VOCAB_PATH = os.path.join(CODE_DIR, "data", "output_vocabulary")

# ── Model output directories ──
STAGE2_DIR = os.path.join(MODELS_ROOT, "stage2", MODEL_NAME)
STAGE3_DIR = os.path.join(MODELS_ROOT, "stage3", MODEL_NAME)

# ── Create folders safely ──
for d in [STAGE2_DIR, STAGE3_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Debug output ──
print("✅ Paths defined (REPRODUCIBLE SETUP)")
print(f"  Base Dir : {BASE_DIR}")
print(f"  Model    : {MODEL_NAME}")
print(f"  Data Dir : {DATA_DIR}")
print(f"  Code Dir : {CODE_DIR}")
print(f"  Models   : {MODELS_ROOT}")

print("\n Verifying dataset files:")
for name, path in [
    ("WI Train", WI_TRAIN),
    ("WI Dev", WI_DEV),
    ("Vocab", VOCAB_PATH),
]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {name}: {path}")

✅ Paths defined!

Verifying dataset files:
  ✅ WI Train: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_train_gector.txt
  ✅ WI Dev: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  ✅ Vocab: /content/gector/data/output_vocabulary


In [ ]:
import os, shutil, json

# 🔹 USER CONFIG (must match your main config cell)
BASE_DIR = "/path_to_project_root"
MODEL_NAME = "mobilebert"

# ── Paths ──
MODELS_ROOT = os.path.join(BASE_DIR, "models")

stage2_checkpoint = os.path.join(MODELS_ROOT, "stage2", MODEL_NAME)

os.makedirs(STAGE2_DIR, exist_ok=True)

files_to_restore = [
    "model_best.th",
    "training_history.json",
    "metrics.json"
]

print("🔄 Restoring Stage 2 best model...")

for f in files_to_restore:
    src = os.path.join(stage2_checkpoint, f)
    dst = os.path.join(STAGE2_DIR, f)

    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f"  ✅ {f} ({size:.1f} MB)")
    else:
        print(f"  ⚠️ {f} not found — skipping")

# ── Show Stage 2 results ──
history_path = os.path.join(STAGE2_DIR, "training_history.json")

if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)

    print(f"\n📊 Stage 2 Results:")
    print(f"   Best epoch    : {history.get('best_epoch', 'N/A')}")
    print(f"   Best val loss : {history.get('best_val_loss', 'N/A')}")
    print(f"   Last epoch    : {history.get('last_epoch', 'N/A')}")

# ── Verify model ──
best_model = os.path.join(STAGE2_DIR, "model_best.th")

if os.path.exists(best_model):
    size = os.path.getsize(best_model) / 1e6
    print(f"\n✅ model_best.th ready ({size:.1f} MB)")
    print("   Stage 3 will load this as pretrained weights!")
else:
    print("\n❌ model_best.th not found!")
    print("   Cannot proceed to Stage 3!")

Restoring Stage 2 best model...
  ✅ model_best.th (109.1 MB)
  ✅ training_history.json (0.0 MB)
  ✅ metrics.json (0.0 MB)

📊 Stage 2 Results:
   Best epoch    : 8
   Best val loss : 0.5548636540770531
   Last epoch    : 13

✅ model_best.th ready (109.1 MB)
   Stage 3 will load this as pretrained weights!


In [6]:
# ── Cell 9: Cache Encoder Models ─────────────────────────────────────────────
from transformers import AutoModel, AutoTokenizer
import torch

print("Pre-downloading all encoder models...")

for model_name in [
    "huawei-noah/TinyBERT_General_4L_312D",
    "albert-base-v2",
    "google/mobilebert-uncased"
]:
    print(f"Downloading {model_name}...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  Done!")

torch.cuda.empty_cache()
print("\n✅ All models cached! Training will not need internet.")


Pre-downloading all encoder models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertModel LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
mobilebert.embeddings.position_ids         | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.dense.weight               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Done!

✅ All models cached! Training will not need internet.


In [ ]:
# ── Cell: Run Stage 3 Training ─────────────────────────────────────────────
import subprocess
import os
import sys

# ⚠️ IMPORTANT:
# The transformer model is controlled by MODEL_NAME (set in config cell).
# To switch models, simply change:
#   MODEL_NAME = "mobilebert" → "albert" or "tinybert"
# No need to modify anything in this cell.

os.chdir(WORK_DIR)

stage3_cmd = [
    "python", "resume_training.py",
    "--train_set",          WI_TRAIN,
    "--dev_set",            WI_DEV,
    "--model_dir",          STAGE3_DIR,
    "--vocab_path",         VOCAB_PATH,
    "--transformer_model",  MODEL_NAME,   # uses selected model
    "--pretrain_folder",    STAGE2_DIR,
    "--pretrain",           "model_best",
    "--batch_size",         "32",
    "--accumulation_size",  "4",
    "--n_epoch",            "25",
    "--patience",           "5",
    "--cold_steps_count",   "0",
    "--lr",                 "1e-5",
    "--skip_correct",       "0",
    "--skip_complex",       "0",
    "--tune_bert",          "1",
    "--tag_strategy",       "keep_one",
    "--lowercase_tokens",   "1",
    "--pieces_per_token",   "5",
    "--tn_prob",            "0.1",
    "--tp_prob",            "0.9",
    "--updates_per_epoch",  "1000",
    "--special_tokens_fix", "0",
]

IMPORTANT_KEYWORDS = [
    "Using encoder", "Vocab loaded", "Train:", "Dev:",
    "Pretrained weights loaded", "Using device",
    "RESUMING", "Resuming", "Training on device",
    "Starting from epoch", "Total epochs", "Batch size",
    "Effective batch", "Updates/epoch", "Patience",
    "EPOCH", "Train loss", "Val loss", "Best epoch",
    "Saved", "improvement", "Patience:", "Summary",
    "TRAINING COMPLETE", "Best val loss", "Total time",
]

print(f"🚀 Starting Stage 3 Training ({MODEL_NAME})...")
print(f"  Train set   : {WI_TRAIN}")
print(f"  Dev set     : {WI_DEV}")
print(f"  Stage2 ckpt : {STAGE2_DIR}")
print(f"  Model dir   : {STAGE3_DIR}")
print()

process = subprocess.Popen(
    stage3_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    line = line.rstrip()
    if "Training epoch" in line:
        sys.stdout.write(f"\r{line}")
        sys.stdout.flush()
    elif any(kw in line for kw in IMPORTANT_KEYWORDS):
        sys.stdout.write(f"\n{line}\n")
        sys.stdout.flush()

process.wait()
print("\nStage 3 Done!")

Starting Stage 3 Training (ALBERT-base-v2)...
  Train set  : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_train_gector.txt
  Dev set    : /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  Stage2 ckpt: /content/models/stage2
  Model dir  : /content/models/stage3


Using encoder: google/mobilebert-uncased

Vocab loaded: 5002 labels, 4 d_tags

Train: 34304 instances | Dev: 4382 instances

Pretrained weights loaded!

Using device: cuda:0

Training on device  : cuda:0

Starting from epoch : 1

Total epochs        : 25

Batch size          : 32

Effective batch     : 128

Updates/epoch       : 1000

Patience            : 5

EPOCH 1/25
Training epoch 1: 100%|██████████| 964/964 [02:19<00:00,  6.91it/s, loss=0.7002, updates=241]
Train loss : 0.7002

Val loss   : 0.6359

💾 Saved 🏆 BEST (epoch 1) → model_best.th

💾 Saved epoch 1 → model_epoch1.th

💾 Saved last (epoch 1) → model_last.th

📊 Epoch 1 Summary:

   Train loss 

In [8]:
import shutil, os, json, zipfile

STAGE3_OUT = "/content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage3/mobilebert"
os.makedirs(STAGE3_OUT, exist_ok=True)

# ── Save individual files to Drive ──
print(f"Saving Stage 3 checkpoint to Drive: {STAGE3_OUT}")
for fname in ["model_best.th", "metrics.json", "training_history.json"]:
    src = os.path.join(STAGE3_DIR, fname)
    dst = os.path.join(STAGE3_OUT, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f"  ✅ {fname} ({size:.1f} MB)")
    else:
        print(f"  ⚠️  {fname} not found — skipping")

# ── Show final metrics ──
metrics_path = os.path.join(STAGE3_DIR, "metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("\nStage 3 final metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

# ── Zip model + vocab + scripts ──
zip_local = "/content/stage3_mobilebert_final.zip"
with zipfile.ZipFile(zip_local, "w") as z:
    # Model weights
    for fname in ["model_best.th", "metrics.json", "training_history.json"]:
        src = os.path.join(STAGE3_DIR, fname)
        if os.path.exists(src):
            z.write(src, fname)
            print(f"✅ Added {fname}")
        else:
            print(f"❌ MISSING: {fname}")

    # Scripts
    for fname in ["predict.py"]:
        src = os.path.join(work_dir, fname)
        if os.path.exists(src):
            z.write(src, fname)
            print(f"✅ Added {fname}")

    # output_vocabulary folder
    for root, dirs, files in os.walk(VOCAB_PATH):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.join("output_vocabulary", os.path.relpath(full_path, VOCAB_PATH))
            z.write(full_path, arcname)
    print("✅ Added output_vocabulary/")

size = os.path.getsize(zip_local) / 1e6
print(f"\n✅ stage3_albert_final.zip ({size:.1f} MB)")

# ── Copy zip to Drive ──
zip_drive = os.path.join(STAGE3_OUT, "stage3_mobilebert_final.zip")
shutil.copy(zip_local, zip_drive)
print(f"✅ Zip saved to Drive: {zip_drive}")

Saving Stage 3 checkpoint to Drive: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage3/mobilebert
  ✅ model_best.th (109.1 MB)
  ✅ metrics.json (0.0 MB)
  ✅ training_history.json (0.0 MB)

Stage 3 final metrics:
  training_duration: 0:22:26
  best_epoch: 4
  best_val_loss: 0.6005
  last_epoch: 9
  total_epochs_trained: 9
  history: [{'epoch': 1, 'train_loss': 0.70016, 'val_loss': 0.635911, 'is_best': True, 'epoch_time_seconds': 146, 'early_stopped': False}, {'epoch': 2, 'train_loss': 0.664609, 'val_loss': 0.623468, 'is_best': True, 'epoch_time_seconds': 147, 'early_stopped': False}, {'epoch': 3, 'train_loss': 0.642841, 'val_loss': 0.622946, 'is_best': True, 'epoch_time_seconds': 145, 'early_stopped': False}, {'epoch': 4, 'train_loss': 0.627404, 'val_loss': 0.600494, 'is_best': True, 'epoch_time_seconds': 146, 'early_stopped': False}, {'epoch': 5, 'train_loss': 0.610642, 'val_loss': 0.608509, 'is_best': False, 'epoch_time_seconds': 146, 'early_stopped': False}, {'epoch': 6,